# Plot variables

In [ ]:
import geopandas as gpd
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib_scalebar.scalebar import ScaleBar
from sklearn import metrics


## Load data

In [ ]:
fi = {}
lc = {}
perf = []

for reduction in ["fa"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = model.feature_importances_
                else:
                    lc[reduction][model_type][cluster] = model.local_coef_
                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,

                        }
                    )
                )

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
'Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem',
 'Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem',
 'Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem',
 'Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem',
 'Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem',
 'Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem',
 'Zaměstnaní - Pracovníci ve službách a prodeji',
 'Zaměstnaní - Řemeslníci a opraváři',
 'Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem',
 'Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví',
 'Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý',
 'Počet osob v bytech celkem  s právním důvodem užívání: družstevní',
 'Počet obyvatel na byt',
 'Počet osob v domech celkem s vlastnictvím:  fyzická osoba',
 'Obyvatelstvo - věk: 0 - 6  - celkem',
 'Obyvatelstvo - věk: 7 - 14  - celkem',
 'Obyvatelstvo - věk: 15 - 24  - celkem',
 'Obyvatelstvo - věk: 45 - 54  - celkem',
 'Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem',
 'Obyvatelstvo - státní občanství: Slovenská republika - celkem',
 'Obyvatelstvo - státní občanství: země EU mimo ČR - celkem',
 'Obyvatelstvo - státní občanství: nezjištěno - celkem',
 'Obyvatelstvo - náboženská víra: bez náboženské víry - celkem',
 'Obyvatelstvo - náboženská víra: neuvedeno - celkem',
 'Obyvatelstvo - s trvalým pobytem - celkem',
 'Obyvatelstvo - rodinný stav: ženatí, vdané - celkem',
 'Obyvatelstvo - rodinný stav: rozvedení - celkem',
 'Obyvatelstvo - rodinný stav: ovdovělí - celkem',
    "geometry",
]
fas = census[selection]

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

## LR

In [ ]:
models = []

for reduction in ["fa"]:
    for model_type in ["lr"]:
        for cluster in [5, 4]:
            path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
            with open(path, "rb") as f:
                model = joblib.load(f)
            f1_values = model.local_metric(metrics.f1_score, average='macro', zero_division=0)

            models.append(
                {
                    "reduction": reduction,
                    "model_type": model_type,
                    "cluster": cluster,
                    "model": model,
                    "f1_values": model.local_metric(metrics.f1_score, average='macro', zero_division=0)
                }
            )

df_models = pd.DataFrame(models)

In [ ]:
morpho_names = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

In [ ]:
global_f1_max = max(
    np.nanmax(row["f1_values"]) for _, row in df_models.iterrows()
)

global_f1_min = min(
    np.nanmin(row["f1_values"]) for _, row in df_models.iterrows()
)


fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flat

for ax, (_, row) in zip(axes, df_models.iterrows(), strict=False):
    model = row["model"]
    cluster = row["cluster"]

    # extract local F1-macro values
    f1_series = row["f1_values"]

    data.plot(
        ax=ax,
        column=f1_series,
        cmap="YlGnBu",
        legend=True,
        vmin=global_f1_min,
        vmax=global_f1_max,
        missing_kwds={"color": "lightgray"},
        legend_kwds={"shrink": 0.6},
    )

    ax.set_title(morpho_names[cluster])
    ax.set_axis_off()
    ax.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
coef_cols = [
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
]

# global scaling per variable
global_max_dict = {
    col: max(
        row["model"].local_coef_[col].max()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

global_min_dict = {
    col: max(
        row["model"].local_coef_[col].min()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}
n_clusters = len(df_models)

In [ ]:
global_max_dict

In [ ]:
global_min_dict

Plot spatial variability of seelected variavles

In [ ]:

def plot_2x4_grid_fixed_scale(coef_col):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()

    # Turn off all axes initially
    for ax in axes:
        ax.axis("off")

    # Plot morphotypes
    for i, (_, row) in enumerate(df_models.iterrows()):
        if i >= 8:
            break

        ax = axes[i]
        model = row["model"]
        cluster = row["cluster"]

        series = model.local_coef_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)

        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )

        ax.set_title(morpho_names[cluster], fontsize=10)
        ax.axis("off")

    # Colorbar in col 4, row 2 (last subplot)
    cax = fig.add_axes((0.82, 0.15, 0.02, 0.25))  # adjust as needed
    sm = mpl.cm.ScalarMappable(
        cmap="coolwarm",
        norm=mpl.colors.Normalize()
    )
    sm.set_array([-1,1])
    fig.colorbar(sm, cax=cax)

    fig.suptitle(coef_col, fontsize=14)
    fig.subplots_adjust(wspace=0.05, hspace=0.15)


In [ ]:
plot_2x4_grid_fixed_scale(coef_cols[0])


In [ ]:
plot_2x4_grid_fixed_scale(coef_cols[1])


In [ ]:
coef_cols = [
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
#    "Obyvatelstvo - s dlouhodobým pobytem - celkem",
]

n_clusters = len(df_models)

fig, axes = plt.subplots(
    n_clusters, len(coef_cols), figsize=(6 * len(coef_cols), 4 * n_clusters)
)


# global scaling per variable
global_max_dict = {
    col: max(
        row["model"].local_coef_[col].max()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

global_min_dict = {
    col: max(
        row["model"].local_coef_[col].min()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

# loop over clusters and variables
for row_i, (_, row) in enumerate(df_models.iterrows()):
    model = row["model"]
    cluster = row["cluster"]

    for col_i, coef_col in enumerate(coef_cols):

        ax = axes[col_i, row_i]

        series = model.local_coef_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)

        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

axes[0][0].set_title("Incoherent Small-Scale Compact Fabric", fontsize=12)
axes[0][0].set_ylabel("Res. in Owner-Occupied Dwellings", fontsize=10)
axes[0][1].set_title("Incoherent Small-Scale Sparse Fabric", fontsize=12)
axes[1][0].set_ylabel("Empl. sector - Agriculture/Forestry/Fishery", fontsize=10)
axes[1][0].add_artist(ScaleBar(1,location="lower left",height_fraction=0.015))



# Colorbar in col 4, row 2 (last subplot)
cax = fig.add_axes((0.95, 0.2, 0.02, 0.6))
sm = mpl.cm.ScalarMappable(
    cmap="coolwarm",
    norm=mpl.colors.Normalize()
)
sm.set_array([-1,1])
fig.colorbar(sm, cax=cax)
#plt.tight_layout()
fig.subplots_adjust(wspace=0, hspace=0)
#fig.align_titles()

#plt.tight_layout()
fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")


In [ ]:
cluster

In [ ]:
coef_cols = [
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
]

n_models = len(df_models)
n_rows = int(np.ceil(n_models / 2))

fig, axes = plt.subplots(n_rows, 4, figsize=(12, 3 * n_rows))



global_max_dict = {
    col: max(
        row["model"].local_coef_[col].max()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

global_min_dict = {
    col: min(
        row["model"].local_coef_[col].min()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}
row_i = 0

for i in range(0, n_models, 2):
    for j in range(2):
        if i + j >= n_models:
            break

        model_row = df_models.iloc[i + j]
        model = model_row["model"]
        cluster = model_row["cluster"]

        for col_i, coef_col in enumerate(coef_cols):
            ax = axes[row_i, j * 2 + col_i]

            series = model.local_coef_[coef_col]
            tmp = data.assign(_coef_tmp=series.values)

            tmp.plot(
                column="_coef_tmp",
                ax=ax,
                cmap="coolwarm",
                vmin=-1.8,
                vmax=1.8,
                legend=False,
                missing_kwds={"color": "lightgray"},
            )

            ax.set_title(
                f"{morpho_names[cluster]}\n{coef_col}",
                fontsize=9
            )

            ax.axis("off")

    row_i += 1

plt.tight_layout()



plot sptial variability of f1 macro

# Composite

In [ ]:
clusters = [1, 2, 3, 4, 5, 7, 8]
reduction = "fa"
model_type = "lr"

# Create a copy of the original data for plotting
plot_data = data.copy()

# Add a column to store the local coefficient values
plot_data["local_coef"] = None

for cluster in clusters:
    # Load model
    model_path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    # Filter original cluster geometries
    compact = data[data["Cluster"] == cluster]

    # Create GeoDataFrame with model results
    compact_model = gpd.GeoDataFrame(
        {
            "geometry": model.geometry,
            "local_coef": model.local_coef_[
                "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý"
            ],
        }
    )

    # Spatial join to map model values onto original geometries
    compacts = gpd.sjoin(compact_model, compact, predicate="within")
    compacts = compacts.drop(columns="index_right")

    # Assign local_coef values back to original geometries
    plot_data.loc[plot_data["Cluster"] == cluster, "local_coef"] = compacts[
        "local_coef"
    ].values
    plot_data["local_coef"] = plot_data["local_coef"].astype(float)
# Now plot all clusters in one map
fig, ax = plt.subplots(figsize=(12, 10))
plot_data.plot(
    column="local_coef",
    cmap="RdYlGn_r",
    legend=True,
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    ax=ax,
)
ax.set_title(
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý"
)
ax.set_axis_off()
plt.show()

In [ ]:
reduction = "fa"
model_type = "lr"

# Create a copy of the original data for plotting
plot_data = data.copy()

# Add a column to store f1_macro values
plot_data["f1_macro"] = None

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    # Load model
    model_path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    # Filter original cluster geometries
    compact = data[data["Cluster"] == cluster]

    # Create GeoDataFrame with model results
    compact_model = gpd.GeoDataFrame(
        {
            "geometry": model.geometry,
            "local_pooled_f1_macro": metrics.f1_score(model.y_pooled_, model.pred_pooled_, average='macro'),
        }
    )

    # Spatial join to map model values onto original geometries
    compacts = gpd.sjoin(compact_model, compact, predicate="within")
    compacts = compacts.drop(columns="index_right")

    # Assign f1_macro values back to original geometries
    plot_data.loc[plot_data["Cluster"] == cluster, "f1_macro"] = compacts[
        "local_pooled_f1_macro"
    ].values

In [ ]:
plot_data["f1_macro"] = plot_data["f1_macro"].astype(float)

In [ ]:
# Now plot all clusters in one map
fig, ax = plt.subplots(figsize=(12, 10))
plot_data.plot(
    column="f1_macro",
    scheme="natural_breaks",
    k=4,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "pink"},
    ax=ax,
)
ax.set_title("Local pooled F1 macro across all clusters")
ax.set_axis_off()
plt.show()

In [ ]:
# Now plot all clusters in one map
fig, ax = plt.subplots(figsize=(12, 10))
plot_data.plot(
    column="f1_macro",
    cmap="YlGnBu",
    legend=False,
    missing_kwds={"color": "pink"},
    ax=ax,
)
ax.set_title("Local pooled F1 macro across all clusters")
ax.set_axis_off()
plt.show()